# AERIVA core: quickstart
Run top to bottom. Set `DATA_DIR` to the folder holding the three challenge files.

In [ ]:
DATA_DIR = 'data'   # folder with flight_data.xlsx, data_international_train.xlsx, data_domestic_train.xlsx
import numpy as np
import pandas as pd
from aeriva import data, features, backtest, scenario, replay
from aeriva.models import L2

## 1. Load, clean, check

In [ ]:
f = data.load_flights(DATA_DIR)
i = data.load_international(DATA_DIR)
d = data.load_domestic(DATA_DIR)
data.integrity_report(f, i, d)

## 2. Build the market × month panel (Appendix A features)

In [ ]:
p = features.build_panel(f, i)
p[['market','month','gn','gn_l12','seats','dln_seats','d_ramadan','d_eid','load_factor','ahr_12','gateway_tier']].tail()

## 3. Rolling back-test: L0 vs L1 vs L2

In [ ]:
origins = pd.period_range('2024-01', '2025-07', freq='M')
bt = backtest.rolling_backtest(p, origins)
backtest.summary(bt).round(3)

In [ ]:
print('80% interval coverage:', round(backtest.interval_coverage(bt), 2))
backtest.market_table(bt).round(3).head(10)

## 4. Fit L2 on all data

In [ ]:
m = L2().fit(p)
print(m.params.round(3))
print('beta 95% range:', np.round(m.beta_ci(), 2))

## 5. Forecast one market-month with an 80% range

In [ ]:
row = p[(p.market == 'UNITED KINGDOM') & (p.month == pd.Period('2025-07', 'M'))]
fc = float(m.predict(row).iloc[0])
past_errors = np.log(bt.loc[bt.h == 1, 'gn'] / bt.loc[bt.h == 1, 'L2'])
lo, hi = backtest.prediction_interval(fc, past_errors)
print(f'forecast {fc:,.0f}  range {lo:,.0f}-{hi:,.0f}  actual {row.gn.iloc[0]:,.0f}')

## 6. Scenario: +20% seats

In [ ]:
r = scenario.seats_change(gn_base=fc, seats_base=float(row.seats.iloc[0]), pct=0.20)
print(f'change {r.delta_central:+,.0f}  (range {r.delta_low:+,.0f} to {r.delta_high:+,.0f})')

## 7. Shock replay: Kuwait 2023

In [ ]:
tbl, _ = replay.replay(p, 'KUWAIT', '2023-07', ['2023-07', '2023-08', '2023-09'])
tbl